# 03 — RAG Üretimi ve Yanıt Değerlendirme (İP-4, İP-6)

Kazanılan erişim sonuçlarını sabit LLM'e bağlar: her sistem için Top-k kaynak + yanıt üretir, yanıtları kaydeder.

**Değişmez kural:** Tüm sistemlerde aynı LLM, prompt (`prompts/rag_tr.txt`), Top-k ve üretim ayarları. Fark yalnızca erişim katmanından gelir (RQ3).

In [ ]:
import sys
sys.path.insert(0, '/content/RAG-proje')
%cd /content/RAG-proje

from src.config import load_config
from src.data import jsonl_oku, jsonl_yaz, korpus_yukle, yonetmelik_parcala
from src.retrieval import Bm25Retriever
from src.generation import RagGenerator

cfg = load_config()
chunklar = yonetmelik_parcala(
    korpus_yukle(cfg['veri']['korpus']),
    boyut=cfg['parcalama']['boyut_karakter'],
    ortusme=cfg['parcalama']['ortusme_karakter'],
)
chunk_metin = {c.chunk_id: c.metin for c in chunklar}
sorular = [s for s in jsonl_oku(cfg['veri']['sorular']) if s.get('bolme', 'test') == 'test']
print(f'{len(sorular)} test sorusu')

In [ ]:
# Üreticiyi yükle (4-bit; ilk seferde model indirilir, ~5 GB)
uretici = RagGenerator(
    model_adi=cfg['uretim']['llm'],
    dort_bit=(cfg['uretim']['yukleme'] == '4bit'),
    maks_yeni_token=cfg['uretim']['maks_yeni_token'],
)
print('LLM hazır:', cfg['uretim']['llm'])

In [ ]:
# Örnek akış: BM25 ile üretim. Dense sistemler için FaissRetriever
# notebook 02'deki kurulumla aynı şekilde eklenir; döngü aynıdır.
sistem = Bm25Retriever(chunklar)
top_k = cfg['erisim']['ana_top_k']

kayitlar = []
for soru in sorular[:5]:  # önce küçük örnek; sonra tümünü açın
    bulunan = sistem.retrieve(soru['soru'], k=top_k)
    kaynaklar = [chunk_metin[cid] for cid, _ in bulunan]
    yanit = uretici.uret(soru['soru'], kaynaklar)
    kayitlar.append({
        'soru_id': soru['soru_id'],
        'sistem': 'bm25',
        'soru': soru['soru'],
        'kaynaklar': [cid for cid, _ in bulunan],
        'yanit': yanit,
        'referans': soru.get('referans_yanit', ''),
    })
    print(soru['soru_id'], '->', yanit[:110], '...')

jsonl_yaz(kayitlar, 'results/yanitlar-bm25-ornek.jsonl')
print('Kaydedildi: results/yanitlar-bm25-ornek.jsonl')

## Yanıt değerlendirme (faithfulness)

- Üretilen her yanıt için kaynaklar kaydedildiği için faithfulness, yanıtın iddialarının kaynaklarda desteklenip desteklenmediğine bakarak ölçülür.
- Pratik yol: `ragas` ile LLM-as-judge (faithfulness + answer correctness). **Judge modeli, üretim LLM'inden farklı ve sabit sürümlü olmalı**; hangi modelin judge olduğu rapora yazılır.
- RQ3 analizi: soru bazında retrieval skoru (notebook 02) ile yanıt doğruluğunu eşleştirip korelasyon bakılır; doğrusal olmayan ilişki beklenen bulgudur (H3).

In [ ]:
# RAGAS değerlendirme iskeleti (judge LLM yapılandırması İP-4'te netleşir)
# from datasets import Dataset
# from ragas import evaluate
# from ragas.metrics import faithfulness, answer_correctness
#
# veri = Dataset.from_list([{
#     'question': k['soru'],
#     'answer': k['yanit'],
#     'contexts': [chunk_metin[c] for c in k['kaynaklar']],
#     'ground_truth': k['referans'],
# } for k in kayitlar])
# sonuc = evaluate(veri, metrics=[faithfulness, answer_correctness])
# print(sonuc)